In [ ]:
!pip install langchain

In [ ]:
!pip install langchain_core

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate(
    messages=[
        ("system", """You are a career decision engine. When the user gives job offers and a goal, output ONLY valid JSON in this exact structure — no markdown, no code fences, no explanation text before or after:

```json
{{
    "goal": "<user's stated long-term goal>",
    "options": [
        {{
            "name": "<option name>",
            "salary": <number>,
            "remote": <true/false>,
            "relocation": "<city or null>",
            "learning": "<Unknown/Better/Worse/Same>"
        }}
    ],
    "decision_factors": [
        "<list of factors relevant to this decision>"
    ],
    "recommendation": "<name of chosen option>",
    "reasoning": "<1-2 sentence explanation tied to the stated goal>"
}}
```

Rules:
- Always output valid, parseable JSON only — nothing else.
- 'options' array can have 2 or more entries depending on how many the user gives.
- 'decision_factors' should reflect what actually matters for this specific decision.
- 'recommendation' must match one of the 'name' values exactly.
- Never hedge in 'reasoning' — give a clear justification, not a balanced view.
- If a field's value is unknown, use 'Unknown' (strings) or null (optional fields)."""),
        ("human", "{user_input}")
    ]
)

In [ ]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.3 MB/s eta 0:00:00


In [ ]:
# create an llm

from langchain_groq import ChatGroq

GROQ_API_KEY="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"

gpt_llm=ChatGroq(
    api_key=GROQ_API_KEY,
    model="openai/gpt-oss-120b",
    temperature=0.5
)

In [ ]:
gpt_llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7f1b24292d50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7f1b02dd77a0>, model_name='openai/gpt-oss-120b', temperature=0.5, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
# creaet an output parser

from langchain_core.output_parsers import JsonOutputParser

json_parser=JsonOutputParser()

In [ ]:
chain= template | gpt_llm | json_parser

In [ ]:
chain

ChatPromptTemplate(input_variables=['user_input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a career decision engine. When the user gives job offers and a goal, output ONLY valid JSON in this exact structure — no markdown, no code fences, no explanation text before or after:\n\n```json\n{{\n    "goal": "<user\'s stated long-term goal>",\n    "options": [\n        {{\n            "name": "<option name>",\n            "salary": <number>,\n            "remote": <true/false>,\n            "relocation": "<city or null>",\n            "learning": "<Unknown/Better/Worse/Same>"\n        }}\n    ],\n    "decision_factors": [\n        "<list of factors relevant to this decision>"\n    ],\n    "recommendation": "<name of chosen option>",\n    "reasoning": "<1-2 sentence explanation tied to the stated goal>"\n}}\n```\n\nRules:\n- Always output valid, parseable JSON o

In [ ]:
chain.invoke({"user_input":"I got two job offers. Company A gives ₹40k but requires relocating to Pune. Company B gives ₹32k, is remote, and has better learning opportunities. I eventually want to work abroad."})

{'goal': 'work abroad',
 'options': [{'name': 'Company A',
   'salary': 40000,
   'remote': False,
   'relocation': 'Pune',
   'learning': 'Unknown'},
  {'name': 'Company B',
   'salary': 32000,
   'remote': True,
   'relocation': None,
   'learning': 'Better'}],
 'decision_factors': ['salary',
  'relocation requirement',
  'remote work flexibility',
  'learning opportunities',
  'alignment with goal to work abroad'],
 'recommendation': 'Company B',
 'reasoning': "Company B's remote setup and superior learning opportunities directly support your aim to work abroad, outweighing the lower salary."}